# Capítulo 05 — Exercícios de pré-treinamento

Notebook de prática para os exercícios 5.1–5.6 de **pré-treinamento em dados sem rótulos**. Vamos usar o conto *The Verdict*, o tokenizador GPT-2 e a implementação GPT dos capítulos anteriores.

> **Execução segura:** os exercícios de amostragem e o treino de um GPT pequeno são leves. O download do GPT-2 pré-treinado (5.5) e do GPT-2 XL (5.6) é opcional e fica desativado por padrão: podem exigir centenas de MB a vários GB, além de TensorFlow.

As células de código serão executadas na ordem. Algumas seções incluem perguntas e espaço para registrar observações.

## 1. Importar bibliotecas necessárias

Usaremos PyTorch, NumPy, `tiktoken` e utilitários locais do capítulo. Não precisamos de `pandas` nem de Hugging Face Transformers para estes exercícios: o material implementa o GPT com PyTorch.

In [1]:
from pathlib import Path
import sys
import tempfile
import numpy as np
import torch
import tiktoken

# Localiza a pasta do código principal mesmo se o kernel iniciou na raiz do workspace.
chapter_dir = None
for base in (Path.cwd(), *Path.cwd().parents):
    for candidate in (
        base / "ch05" / "01_main-chapter-code",
        base / "Public" / "ch05" / "01_main-chapter-code",
    ):
        if (candidate / "previous_chapters.py").is_file():
            chapter_dir = candidate.resolve()
            break
    if chapter_dir is not None:
        break

if chapter_dir is None:
    raise FileNotFoundError(
        "Não encontrei ch05/01_main-chapter-code. Abra o workspace correto no VS Code "
        "ou ajuste chapter_dir para a localização do repositório."
    )

sys.path.insert(0, str(chapter_dir))
from previous_chapters import GPTModel, create_dataloader_v1, generate_text_simple

print("PyTorch:", torch.__version__)
print("Pasta do capítulo:", chapter_dir)
print("Dispositivo disponível:", "CUDA" if torch.cuda.is_available() else "CPU")

PyTorch: 2.14.1+cpu
Pasta do capítulo: /home/danielanderson.silva.copin/Downloads/transformer-model-engineering
Public/ch05/01_main-chapter-code
Dispositivo disponível: CPU


## 2. Carregar e visualizar o dataset

O capítulo usa *The Verdict*, um conto curto. Tentamos primeiro encontrar uma cópia local em `ch02`; se não houver, baixamos somente o arquivo de texto público.

In [3]:
import requests

# Mantenha o dataset fora do repositório, para que a execução não crie arquivos extras no projeto.
local_candidates = [
    chapter_dir.parents[2] / "ch02" / "01_main-chapter-code" / "the-verdict.txt",
    Path(tempfile.gettempdir()) / "llms_from_scratch_the-verdict.txt",
]
data_path = next((p for p in local_candidates if p.is_file()), None)

if data_path is not None:
    text_data = data_path.read_text(encoding="utf-8")
    print("Dataset lido de:", data_path)
else:
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    data_path = Path(tempfile.gettempdir()) / "llms_from_scratch_the-verdict.txt"
    data_path.write_text(text_data, encoding="utf-8")
    print("Dataset baixado para cache temporário:", data_path)

print("Caracteres:", len(text_data))
print("\nInício do texto:\n", text_data[:500])
print("\nFinal do texto:\n", text_data[-300:])

Dataset baixado para cache temporário: /tmp/llms_from_scratch_the-verdict.txt
Caracteres: 20479

Início do texto:
 I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it'

Final do texto:
  brings me anywhere near him is that I knew enough to leave off?"

He stood up and laid his hand on my shoulder with a laugh. "Only the irony of it is that I _am_ still painting--since Grindle's doing it for me! The Strouds stand alone, and happen once--but there's no exterminating our kind of art."


## 3. Pré-processamento dos dados

Tokenizamos com o vocabulário GPT-2 e dividimos o texto em 90% para treino e 10% para validação. Cada alvo é a sequência de entrada deslocada em um token. Usamos contexto 64 para tornar os exemplos leves.

In [4]:
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = tokenizer.encode(text_data, allowed_special={"<|endoftext|>"})
print("Tokens no texto:", len(token_ids))

train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_text, val_text = text_data[:split_idx], text_data[split_idx:]

CONTEXT_LENGTH = 64
BATCH_SIZE = 2
train_loader = create_dataloader_v1(
    train_text, batch_size=BATCH_SIZE, max_length=CONTEXT_LENGTH,
    stride=CONTEXT_LENGTH, drop_last=True, shuffle=True, num_workers=0,
)
val_loader = create_dataloader_v1(
    val_text, batch_size=BATCH_SIZE, max_length=CONTEXT_LENGTH,
    stride=CONTEXT_LENGTH, drop_last=False, shuffle=False, num_workers=0,
)

input_batch, target_batch = next(iter(train_loader))
print("Lotes de treino:", len(train_loader), "| lotes de validação:", len(val_loader))
print("Forma de entrada:", tuple(input_batch.shape))
print("Forma do alvo:   ", tuple(target_batch.shape))
print("Exemplo de entrada:", tokenizer.decode(input_batch[0].tolist()))
print("Alvo deslocado:   ", tokenizer.decode(target_batch[0].tolist()))

Tokens no texto: 5145
Lotes de treino: 36 | lotes de validação: 4
Forma de entrada: (2, 64)
Forma do alvo:    (2, 64)
Exemplo de entrada:  she sent for you?"

"Yes--quite insensible to the irony. She wanted him vindicated--and by me!"

He laughed again, and threw back his head to look up at the sketch of the donkey. "There were days when I couldn't look at that thing--couldn't
Alvo deslocado:     sent for you?"

"Yes--quite insensible to the irony. She wanted him vindicated--and by me!"

He laughed again, and threw back his head to look up at the sketch of the donkey. "There were days when I couldn't look at that thing--couldn't face


## 4. Construir o modelo Transformer

Importamos a arquitetura GPT da implementação local do capítulo anterior e criamos uma versão pequena apenas para fins didáticos. Ela é inicializada do zero, portanto não se deve esperar texto coerente antes do treino.

In [5]:
GPT_CONFIG = {
    "vocab_size": 50257,
    "context_length": CONTEXT_LENGTH,
    "emb_dim": 64,
    "n_heads": 4,
    "n_layers": 2,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG).to(device)
model.eval()

with torch.no_grad():
    sample_logits = model(input_batch.to(device))
print("Parâmetros do modelo:", sum(p.numel() for p in model.parameters()))
print("Forma dos logits (lote, tokens, vocabulário):", tuple(sample_logits.shape))

Parâmetros do modelo: 6536704
Forma dos logits (lote, tokens, vocabulário): (2, 64, 50257)


## 5. Treinar o modelo

Para demonstrar o ciclo de treino sem transformar a sessão numa execução longa, faremos cinco atualizações de um GPT pequeno. Isso ilustra o processo; **não equivale** ao pré-treinamento completo de GPT-2 apresentado no capítulo.

In [6]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    return torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()
    )


def mean_loss(data_loader, model, device, max_batches=2):
    model.eval()
    losses = []
    with torch.no_grad():
        for i, (inputs, targets) in enumerate(data_loader):
            if i >= max_batches:
                break
            losses.append(calc_loss_batch(inputs, targets, model, device).item())
    model.train()
    return float(np.mean(losses)) if losses else float("nan")


optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)
initial_train_loss = mean_loss(train_loader, model, device)
initial_val_loss = mean_loss(val_loader, model, device)

model.train()
train_iter = iter(train_loader)
for step in range(5):
    try:
        inputs, targets = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        inputs, targets = next(train_iter)
    optimizer.zero_grad(set_to_none=True)
    loss = calc_loss_batch(inputs, targets, model, device)
    loss.backward()
    optimizer.step()
    print(f"Atualização {step + 1}/5 — loss do lote: {loss.item():.3f}")

trained_train_loss = mean_loss(train_loader, model, device)
trained_val_loss = mean_loss(val_loader, model, device)
print(f"\nLoss inicial (treino/validação): {initial_train_loss:.3f} / {initial_val_loss:.3f}")
print(f"Loss após 5 passos (treino/validação): {trained_train_loss:.3f} / {trained_val_loss:.3f}")

Atualização 1/5 — loss do lote: 11.054
Atualização 2/5 — loss do lote: 10.987
Atualização 3/5 — loss do lote: 10.953
Atualização 4/5 — loss do lote: 11.006
Atualização 5/5 — loss do lote: 10.918

Loss inicial (treino/validação): 10.979 / 10.987
Loss após 5 passos (treino/validação): 10.951 / 10.947


## 6. Avaliar o modelo

A métrica central do capítulo é *cross-entropy*; sua exponencial é a perplexidade. Também mostramos acurácia de próximo token para uma amostra, mas ela não substitui a loss durante o treino.

In [7]:
val_loss = mean_loss(val_loader, model, device, max_batches=4)
val_perplexity = float(np.exp(val_loss))

model.eval()
with torch.no_grad():
    eval_inputs, eval_targets = next(iter(val_loader))
    eval_logits = model(eval_inputs.to(device))
    predictions = eval_logits.argmax(dim=-1).cpu()
    next_token_accuracy = (predictions == eval_targets).float().mean().item()
model.train()

print(f"Validation cross-entropy (até 4 lotes): {val_loss:.3f}")
print(f"Validation perplexity: {val_perplexity:.2f}")
print(f"Acurácia de próximo token (1 lote): {next_token_accuracy:.2%}")

Validation cross-entropy (até 4 lotes): 10.955
Validation perplexity: 57233.97
Acurácia de próximo token (1 lote): 0.00%


## 7. Executar exercícios do capítulo 05

### Exercício 5.1 — temperatura e probabilidade de amostragem

Comparamos as distribuições produzidas por três temperaturas e estimamos quantas vezes “pizza” aparece em 1.000 amostras. A frequência é uma estimativa; a probabilidade da softmax é o valor exato para esses logits.

In [8]:
toy_vocab = {
    "closer": 0, "every": 1, "effort": 2, "forward": 3,
    "inches": 4, "moves": 5, "pizza": 6, "toward": 7, "you": 8,
}
toy_logits = torch.tensor([4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79])
temperatures = [1.0, 0.1, 5.0]
pizza_id = toy_vocab["pizza"]

for temperature in temperatures:
    probabilities = torch.softmax(toy_logits / temperature, dim=0)
    torch.manual_seed(123)
    draws = torch.multinomial(probabilities, num_samples=1000, replacement=True)
    observed = int((draws == pizza_id).sum())
    exact = probabilities[pizza_id].item()
    print(
        f"T={temperature:>3}: pizza apareceu {observed}/1000 ({observed/10:.1f}%). "
        f"Probabilidade softmax: {exact:.4%}"
    )

T=1.0: pizza apareceu 0/1000 (0.0%). Probabilidade softmax: 0.0101%
T=0.1: pizza apareceu 0/1000 (0.0%). Probabilidade softmax: 0.0000%
T=5.0: pizza apareceu 41/1000 (4.1%). Probabilidade softmax: 4.2998%


### Exercício 5.2 — experimentar temperature e top-k

A função abaixo aplica os dois controles à geração. Como o GPT deste notebook recebeu apenas algumas atualizações, os resultados servem para observar o mecanismo, não a qualidade linguística.

In [9]:
def generate_with_sampling(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)[:, -1, :]
        if top_k is not None:
            k = min(top_k, logits.size(-1))
            top_values, top_indices = torch.topk(logits, k)
            filtered = torch.full_like(logits, -torch.inf)
            filtered.scatter_(1, top_indices, top_values)
            logits = filtered
        if temperature <= 0:
            next_id = logits.argmax(dim=-1, keepdim=True)
        else:
            probs = torch.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, next_id), dim=1)
    return idx


prompt = "Every effort moves you"
prompt_ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
for temperature, top_k in [(0.3, 5), (1.2, 20)]:
    generated = generate_with_sampling(
        model, prompt_ids, max_new_tokens=20, context_size=CONTEXT_LENGTH,
        temperature=temperature, top_k=top_k,
    )
    print(f"T={temperature}, top_k={top_k}: {tokenizer.decode(generated[0].cpu().tolist())}")

T=0.3, top_k=5: Every effort moves you progressively series fenceedu Sheridan pounded rapesSON proc Petitionettaboa®engeance Goatothoveredarant lease�
T=1.2, top_k=20: Every effort moves you Rodrigasca applaudSanders WorthAIR mothersicone Lysavagewb dope tripled Brus Rog practition reunion then TalibanIt


### Exercício 5.3 — comportamento determinístico

Com `temperature=0`, escolhemos sempre o maior logit. Com `top_k=1`, sobra apenas um token possível; as duas estratégias devem produzir a mesma sequência para o mesmo modelo e prompt.

In [10]:
greedy_ids = generate_with_sampling(
    model, prompt_ids, max_new_tokens=25, context_size=CONTEXT_LENGTH,
    temperature=0.0, top_k=None,
)
top1_ids = generate_with_sampling(
    model, prompt_ids, max_new_tokens=25, context_size=CONTEXT_LENGTH,
    temperature=1.0, top_k=1,
)
print("Mesmo resultado:", torch.equal(greedy_ids, top1_ids))
print("Saída:", tokenizer.decode(greedy_ids[0].cpu().tolist()))
assert torch.equal(greedy_ids, top1_ids)

Mesmo resultado: True
Saída: Every effort moves you progressivelyatham bambooetics granting VID Vol ChaserMalley choirinessiably classmatesClearly dictates Hallow conscienceremote Courier Neon user parser Theo Agents emailed


### Exercício 5.4 — continuar o pré-treinamento

Salvamos os pesos do modelo **e** o estado do AdamW, recarregamos ambos e fazemos mais uma atualização. O arquivo fica em uma pasta temporária e é removido ao final.

In [11]:
with tempfile.TemporaryDirectory() as tmp_dir:
    checkpoint_path = Path(tmp_dir) / "checkpoint_toy_gpt.pth"
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, checkpoint_path)

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    resumed_model = GPTModel(GPT_CONFIG).to(device)
    resumed_model.load_state_dict(checkpoint["model_state_dict"])
    resumed_optimizer = torch.optim.AdamW(
        resumed_model.parameters(), lr=4e-4, weight_decay=0.1
    )
    resumed_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    resumed_model.train()
    resumed_optimizer.zero_grad(set_to_none=True)
    resume_inputs, resume_targets = next(iter(train_loader))
    resume_loss = calc_loss_batch(resume_inputs, resume_targets, resumed_model, device)
    resume_loss.backward()
    resumed_optimizer.step()

print(f"Checkpoint restaurado; loss do passo retomado: {resume_loss.item():.3f}")

Checkpoint restaurado; loss do passo retomado: 10.994


### Exercício 5.5 — loss do GPT-2 pré-treinado

Este exercício baixa os pesos originais do GPT-2 124M e calcula a loss em treino e validação. O download é grande e requer TensorFlow; por segurança, a flag começa desligada. Ative-a somente quando quiser baixar os arquivos.

In [12]:
RUN_GPT2_PRETRAINED = False  # Altere para True para baixar e carregar os pesos (~500 MB + dependências)

if RUN_GPT2_PRETRAINED:
    from gpt_download import download_and_load_gpt2
    from gpt_generate import load_weights_into_gpt

    pretrained_config = {
        "vocab_size": 50257, "context_length": 1024, "emb_dim": 768,
        "n_heads": 12, "n_layers": 12, "drop_rate": 0.0, "qkv_bias": True,
    }
    settings, params = download_and_load_gpt2("124M", models_dir=str(chapter_dir / "gpt2"))
    pretrained_gpt = GPTModel(pretrained_config).to(device)
    load_weights_into_gpt(pretrained_gpt, params)
    pretrained_gpt.eval()

    # Contexto 256 limita a memória dos tensores de logits; o GPT-2 aceita até 1024.
    pretrained_train_loader = create_dataloader_v1(
        train_text, batch_size=1, max_length=256, stride=256,
        drop_last=True, shuffle=False, num_workers=0,
    )
    pretrained_val_loader = create_dataloader_v1(
        val_text, batch_size=1, max_length=256, stride=256,
        drop_last=False, shuffle=False, num_workers=0,
    )
    print("Loss treino:", mean_loss(pretrained_train_loader, pretrained_gpt, device, max_batches=4))
    print("Loss validação:", mean_loss(pretrained_val_loader, pretrained_gpt, device, max_batches=4))
else:
    print("Exercício 5.5 não executado: RUN_GPT2_PRETRAINED=False (nenhum peso foi baixado).")

Exercício 5.5 não executado: RUN_GPT2_PRETRAINED=False (nenhum peso foi baixado).


### Exercício 5.6 — testar um modelo maior

GPT-2 XL tem cerca de 1,56 bilhão de parâmetros. O download e a inferência podem exigir vários GB de armazenamento e memória; a execução permanece desligada. Use uma máquina com recursos suficientes antes de ativar a flag.

In [13]:
RUN_GPT2_XL = False  # Ative somente se tiver armazenamento e RAM/VRAM suficientes.

if RUN_GPT2_XL:
    from gpt_download import download_and_load_gpt2
    from gpt_generate import load_weights_into_gpt

    xl_config = {
        "vocab_size": 50257, "context_length": 1024, "emb_dim": 1600,
        "n_heads": 25, "n_layers": 48, "drop_rate": 0.0, "qkv_bias": True,
    }
    xl_settings, xl_params = download_and_load_gpt2("1558M", models_dir=str(chapter_dir / "gpt2"))
    gpt_xl = GPTModel(xl_config).to(device)
    load_weights_into_gpt(gpt_xl, xl_params)
    gpt_xl.eval()
    xl_prompt = torch.tensor(tokenizer.encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
    xl_result = generate_with_sampling(
        gpt_xl, xl_prompt, max_new_tokens=20, context_size=xl_config["context_length"],
        temperature=0.8, top_k=50,
    )
    print(tokenizer.decode(xl_result[0].cpu().tolist()))
else:
    print("Exercício 5.6 não executado: RUN_GPT2_XL=False (nenhum arquivo grande foi baixado).")

Exercício 5.6 não executado: RUN_GPT2_XL=False (nenhum arquivo grande foi baixado).


## Registro de observações

- **5.1:** A frequência observada de “pizza” ficou próxima da probabilidade calculada? Por que pode haver diferença?
- **5.2:** Como a temperatura e `top_k` mudaram a diversidade do texto?
- **5.3:** A verificação de determinismo passou?
- **5.4:** Por que salvar o estado do otimizador, além dos pesos do modelo, ajuda a retomar o treino?
- **5.5–5.6:** Execute somente se decidir baixar os pesos e tiver dependências e recursos suficientes.

> Este notebook usa cinco passos de treino e um modelo pequeno para demonstração. Para reproduzir os valores do capítulo, use `ch05.ipynb` e seus parâmetros originais.